In [1]:
import os
import sys

root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import numpy as np
import pytorch_lightning as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from syn_project.utils_train import *
from shimmer_ssd.config import LoadedDomainConfig, DomainModuleVariant
from torch.optim.lr_scheduler import OneCycleLR

PATH_DATASETS = os.environ.get("PATH_DATASETS", ".")
BATCH_SIZE = 256 if torch.cuda.is_available() else 64
NUM_WORKERS = int(os.cpu_count() / 2)

In [2]:
import torch.nn as nn
class LatentDiscriminator(nn.Module):
    def __init__(self, workspace_dim: int):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(workspace_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, z):
        return self.model(z)

In [3]:
from shimmer import GlobalWorkspace2Domains, GlobalWorkspaceBase
from torch.optim import AdamW

class AdversarialGlobalWorkspace(GlobalWorkspace2Domains):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # On ajoute le discriminateur
        self.discriminator = LatentDiscriminator(self.workspace_dim)
        self.adversarial_loss = nn.BCELoss()
        
        # Important pour Lightning lors de l'utilisation de plusieurs optimiseurs
        self.automatic_optimization = False

    def training_step(self, batch, batch_idx):
        # On récupère les deux optimiseurs (Générateur et Discriminateur)
        opt_g, opt_d = self.optimizers()

        # 1. Préparation des latents
        # On encode les domaines pour obtenir les représentations latentes
        domain_latents = self.encode_domains(batch)
        gw_latents = self.encode(domain_latents)
        
        # Supposons que tu compares le domaine "A" (faussaire) au domaine "B" (réel)
        # Il faudra adapter les clés selon tes frozensets
        z_fake = gw_latents[frozenset({'attr','v_latents'})]["attr"]
        z_real = gw_latents[frozenset({'attr','v_latents'})]["v_latents"]

        # --- ENTRAÎNEMENT DU DISCRIMINATEUR ---
        self.toggle_optimizer(opt_d)
        
        # Score sur le réel (doit tendre vers 1)
        d_real = self.discriminator(z_real.detach()) # .detach() car on ne veut pas update l'encodeur B ici
        loss_d_real = self.adversarial_loss(d_real, torch.ones_like(d_real))

        # Score sur le faux (doit tendre vers 0)
        d_fake = self.discriminator(z_fake.detach())
        loss_d_fake = self.adversarial_loss(d_fake, torch.zeros_like(d_fake))

        loss_d = (loss_d_real + loss_d_fake) / 2
        
        self.manual_backward(loss_d)
        opt_d.step()
        opt_d.zero_grad()
        self.untoggle_optimizer(opt_d)

        # --- ENTRAÎNEMENT DU GÉNÉRATEUR (Modèle Global Workspace) ---
        self.toggle_optimizer(opt_g)
        
        # Perte de base du Global Workspace (demi-cycles, etc.)
        loss_output = self.loss_mod.step(batch, domain_latents, mode="train")
        gw_loss = loss_output.loss
        
        # Perte Adversariale (on veut que le discriminateur croie que z_fake est vrai)
        d_fake_g = self.discriminator(z_fake)
        g_adv_loss = self.adversarial_loss(d_fake_g, torch.ones_like(d_fake_g))
        
        # On combine (tu peux ajouter un poids lambda ici)
        total_g_loss = gw_loss + g_adv_loss
        
        self.manual_backward(total_g_loss)
        opt_g.step()
        opt_g.zero_grad()
        self.untoggle_optimizer(opt_g)

        # Logging
        self.log("train/loss_d", loss_d, prog_bar=True)
        self.log("train/loss_g_adv", g_adv_loss, prog_bar=True)
        self.log("train/gw_loss", gw_loss, prog_bar=True)

    def configure_optimizers(self):
        # Optimiseur pour le workspace (Générateur)
        opt_g = AdamW(self.parameters(), lr=self.optim_lr, weight_decay=self.optim_weight_decay)
        
        # Optimiseur pour le discriminateur
        opt_d = AdamW(self.discriminator.parameters(), lr=self.optim_lr)
        
        return [opt_g, opt_d], []

In [4]:
from shimmer_ssd.modules.domains import load_pretrained_domains
from torch.optim import Optimizer

root_path = get_project_root()
    # Set up domain configurations

training_params = get_training_params("syn",  "basic_biased_00")

checkpoint_path = Path(f"{root_path}/checkpoints")
config = training_params["config"]
hparams = training_params["hparams"] if "hparams" in training_params else {"temperature": 1, "alpha": 1}

domains = [
        LoadedDomainConfig(
            domain_type=DomainModuleVariant.v_latents,
            checkpoint_path= checkpoint_path / "domain_v.ckpt"
        ),
        LoadedDomainConfig(
            domain_type=DomainModuleVariant.attr_legacy_no_color,
            checkpoint_path=checkpoint_path / "domain_attr.ckpt",
            args=hparams,
        ),
    ]


domain_modules, gw_encoders, gw_decoders = load_pretrained_domains(
        domains,
        config.global_workspace.latent_dim,
        config.global_workspace.encoders.hidden_dim,
        config.global_workspace.encoders.n_layers,
        config.global_workspace.decoders.hidden_dim,
        config.global_workspace.decoders.n_layers,
)

custom_weights = {
            "cycle_attr_through_v_latents_loss_attr": 0.0,
            "cycle_attr_through_v_latents_loss_cat": 0.0,
            "cycle_v_latents_through_attr": 0.0,
            "demi_cycle_attr": 1.0,
            "demi_cycle_v_latents": 1.0,
            "translation_v_latents_to_attr_loss_attr": 1.0,
            "translation_v_latents_to_attr_loss_cat": 1.0,
            "translation_attr_to_v_latents": 0.0,
            "contrastive_loss": 0.0,
            "shape_loss": 1.0
        }

noise = {"mean": 0.0, "std": 0.0}

def get_scheduler(optimizer: Optimizer, scheduler_type: str = "onecycle"):
        if scheduler_type == "onecycle":
            return OneCycleLR(optimizer, config.training.optim.max_lr, 
                          int(config.training.max_steps), 
                          pct_start=config.training.optim.pct_start, div_factor=.38, final_div_factor=5)
        elif scheduler_type == "linear":
            from torch.optim.lr_scheduler import LinearLR
            return LinearLR(
                optimizer,
                start_factor=1.0,
                end_factor=0.1,
                total_iters=config.training.max_steps
                )
        elif scheduler_type == "cosine":
            from torch.optim.lr_scheduler import CosineAnnealingLR
            return CosineAnnealingLR(
                optimizer,
                T_max=config.training.max_steps,
                eta_min=config.training.optim.lr / 100
            )
        else:
            print(f"Scheduler type {scheduler_type} not supported")
            return None

lr = config.training.optim.lr

global_workspace = AdversarialGlobalWorkspace(
        domain_mods= domain_modules,
        gw_encoders = gw_encoders,
        gw_decoders = gw_decoders,
        workspace_dim = config.global_workspace.latent_dim,
        loss_coefs = config.global_workspace.loss_coefficients,
        optim_lr=lr,
        optim_weight_decay = config.training.optim.weight_decay,
        scheduler=get_scheduler,
    )


/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/lightning/fabric/utilities/cloud_io.py:73: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


In [5]:
trainer = Trainer(
        max_epochs=150,
        default_root_dir=config.default_root_dir,
        precision=config.training.precision,
        accelerator=config.training.accelerator,
        devices=config.training.devices,
        reload_dataloaders_every_n_epochs=1
    )


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


In [6]:
domain_classes = get_default_domains(["v_latents", "attr"])
data_module = SimpleShapesDataModule(
        REGULAR_DATASET_PATH,
        domain_classes,
        config.domain_proportions,
        batch_size=config.training.batch_size,
        num_workers=config.training.num_workers,
        seed=config.seed,
        domain_args=config.domain_data_args,
        collate_fn=custom_collate_factory(True),
    )

In [ ]:
trainer.fit(global_workspace, data_module)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


┏━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name             ┃ Type                  ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ gw_mod           │ GWModule              │  6.3 M │ train │     0 │
│ 1 │ selection_mod    │ SingleDomainSelection │      0 │ train │     0 │
│ 2 │ loss_mod         │ GWLosses2Domains      │  6.3 M │ train │     0 │
│ 3 │ discriminator    │ LatentDiscriminator   │ 36.4 K │ train │     0 │
│ 4 │ adversarial_loss │ BCELoss               │      0 │ train │     0 │
└───┴──────────────────┴───────────────────────┴────────┴───────┴───────┘

Trainable params: 520 K                                                                                            
Non-trainable params: 5.8 M                                                                                        
Total params: 6.3 M                                                                                                
Total estimated model params size (MB): 25                                                                         
Modules in train mode: 51                                                                                          
Modules in eval mode: 35                                                                                           
Total FLOPs: 0

Output()

/home/lucas/.cache/pypoetry/virtualenvs/alexis-n7zQ69N0-py3.11/lib/python3.11/site-packages/lightning/pytorch/loops
/fit_loop.py:534: Found 35 module(s) in eval mode at the start of training. This may lead to unexpected behavior 
during training. If this is intentional, you can ignore this warning.